# Holdout Eval: SimPO (Tuned Hyperparameters)

Greedy generation on the same 100 held-out CharXiv reasoning questions every other system in this project was benchmarked on, for the winning config from the `simpo-sweep` kernel's local lr x beta sweep. Uses the exact same extract_final_answer + normalize_answer exact-match scoring as every other holdout number in this project.


In [ ]:
# Cell 1: Pin a known-good stack (identical to evaluate_holdout.ipynb's proven setup)
import subprocess
import sys

subprocess.run(
    [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao', 'transformers', 'peft', 'accelerate'],
    check=False,
)
cmd = [
    sys.executable, '-m', 'pip', 'install', '-q',
    'torch==2.5.1', 'torchvision==0.20.1',
    '--index-url', 'https://download.pytorch.org/whl/cu124',
]
rc = subprocess.run(cmd).returncode
if rc != 0:
    print('Primary torch install failed; fallback: --no-deps + nvidia-cudnn-cu12==9.1.1.17')
    subprocess.run(cmd + ['--no-deps'], check=True)
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        'nvidia-cudnn-cu12==9.1.1.17',
        'nvidia-cublas-cu12', 'nvidia-cuda-runtime-cu12', 'nvidia-cuda-nvrtc-cu12',
        'nvidia-cufft-cu12', 'nvidia-curand-cu12', 'nvidia-cusolver-cu12',
        'nvidia-cusparse-cu12', 'nvidia-nccl-cu12', 'nvidia-nvtx-cu12',
        'triton', 'filelock', 'fsspec', 'jinja2', 'networkx', 'sympy', 'typing-extensions',
    ], check=True)
subprocess.run(
    [
        sys.executable, '-m', 'pip', 'install', '-q',
        'transformers==4.49.0', 'peft==0.14.0', 'accelerate==1.2.1',
        'qwen-vl-utils==0.0.14', 'pillow',
    ],
    check=True,
)

import torch
import transformers
import peft

print(f'Active PyTorch: {torch.__version__}')
print(f'transformers: {transformers.__version__}, peft: {peft.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU required for holdout evaluation.')
props = torch.cuda.get_device_properties(0)
cc = torch.cuda.get_device_capability(0)
print(f'GPU: {props.name}, cc={cc}, VRAM={props.total_memory / 1e9:.1f} GB')
if cc[0] < 6:
    raise RuntimeError(f'GPU compute capability {cc} is too old.')
print('Packages successfully configured.')


In [ ]:
# Cell 2: Checkout repo, locate the winning adapter, load holdout questions
import json
import os
import re
import subprocess
import sys
from pathlib import Path

import torch
from peft import PeftModel
from qwen_vl_utils import process_vision_info
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

repo_dir = Path('/tmp/chart-prm')
if repo_dir.exists():
    subprocess.run(['rm', '-rf', str(repo_dir)], check=True)
subprocess.run(['git', 'clone', 'https://github.com/yahorlahunovich/chart-prm.git', str(repo_dir)], check=True)
os.chdir(repo_dir)
sys.path.insert(0, str(repo_dir / 'src'))

from chart_prm.generator import build_generation_prompt

MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'
SPLIT_PATH = Path('data/splits/eval_reasoning_ids.json')
QUESTIONS_PATH = Path('data/CharXiv/data/reasoning_val.json')
IMAGES_DIR = Path('data/CharXiv/images')
OUTPUT_PATH = Path('/kaggle/working/simpo_tuned_holdout_generations.jsonl')
SUMMARY_PATH = Path('/kaggle/working/simpo_tuned_holdout_accuracy.json')

print('Available /kaggle/input entries:', sorted(p.name for p in Path('/kaggle/input').glob('*')) if Path('/kaggle/input').exists() else [])
adapter_candidates = sorted({p.parent for p in Path('/kaggle/input').rglob('adapter_config.json')}) if Path('/kaggle/input').exists() else []
# The sweep kernel's output intentionally includes every config's adapter (for a full honest
# record, not just the winner) plus a copy of the winner at a directory literally named
# qwen_vl_simpo_tuned_adapter -- prefer that exact match, since there are several adapters
# here by design, not just one.
named_winner = [p for p in adapter_candidates if p.name == 'qwen_vl_simpo_tuned_adapter']
if named_winner:
    adapter_path = named_winner[0]
elif len(adapter_candidates) == 1:
    adapter_path = adapter_candidates[0]
else:
    raise RuntimeError(
        f'Expected a directory named qwen_vl_simpo_tuned_adapter or exactly 1 adapter under '
        f'/kaggle/input, found {len(adapter_candidates)}: {adapter_candidates}'
    )
print(f'simpo_tuned adapter: {adapter_path}')

winning_config_path = adapter_path / 'winning_config.json'
if winning_config_path.exists():
    print('Winning config used for this adapter:')
    print(winning_config_path.read_text(encoding='utf-8'))

if not torch.cuda.is_available():
    raise RuntimeError('Evaluation requires a CUDA GPU.')

with SPLIT_PATH.open(encoding='utf-8') as handle:
    eval_ids = [str(value) for value in json.load(handle)]
with QUESTIONS_PATH.open(encoding='utf-8') as handle:
    question_data = json.load(handle)

missing_image_ids = [qid for qid in eval_ids if not list(IMAGES_DIR.glob(f'{qid}.*'))]
if missing_image_ids:
    print(f'Downloading {len(missing_image_ids)} missing holdout images...')
    subprocess.run([sys.executable, 'scripts/data_prep/download_images.py', '--ids-file', str(SPLIT_PATH)], check=True)

print(f'Holdout size: {len(eval_ids)}')


In [ ]:
# Cell 3: Load base model + tuned SimPO adapter
base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    attn_implementation='sdpa',
    device_map={'': 0},
    low_cpu_mem_usage=True,
)
processor = AutoProcessor.from_pretrained(MODEL_ID)

model = PeftModel.from_pretrained(base_model, str(adapter_path), adapter_name='simpo_tuned')
model.eval()
print('Loaded base model with simpo_tuned adapter.')


In [ ]:
# Cell 4: Generate on all 100 holdout questions
FINAL_ANSWER_RES = [
    re.compile(r'Final Answer:\s*(.+)', re.IGNORECASE | re.DOTALL),
    re.compile(r'\*\*Final Answer\*\*:\s*(.+)', re.IGNORECASE | re.DOTALL),
    re.compile(r'Therefore,? the answer is:?\s*(.+)', re.IGNORECASE | re.DOTALL),
]


def extract_final_answer(text: str) -> str:
    for pattern in FINAL_ANSWER_RES:
        matches = pattern.findall(text or '')
        if matches:
            answer = matches[-1].strip().splitlines()[0].strip().strip('"\'`')
            return answer
    return ''


def normalize_answer(text: str) -> str:
    return re.sub(r'\s+', ' ', (text or '').strip().lower())


def generate_response(active_model, image_path: Path, question: str) -> str:
    prompt = build_generation_prompt(question)
    messages = [{
        'role': 'user',
        'content': [
            {'type': 'image', 'image': str(image_path)},
            {'type': 'text', 'text': prompt},
        ],
    }]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors='pt',
    ).to('cuda')
    with torch.inference_mode():
        generated_ids = active_model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
            use_cache=True,
        )
    generated_ids = generated_ids[:, inputs.input_ids.shape[1]:]
    return processor.batch_decode(
        generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]


def load_done_ids(path: Path) -> set:
    done = set()
    if not path.exists():
        return done
    with path.open(encoding='utf-8') as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            done.add(json.loads(line)['question_id'])
    return done


done_ids = load_done_ids(OUTPUT_PATH)
print(f'Resuming with {len(done_ids)} completed questions.')

system = 'simpo_tuned'
with OUTPUT_PATH.open('a', encoding='utf-8') as handle:
    for idx, question_id in enumerate(eval_ids):
        if question_id in done_ids:
            continue
        record = question_data.get(question_id)
        if record is None:
            raise KeyError(f'Held-out question {question_id} missing from reasoning_val.json')
        image_matches = sorted(IMAGES_DIR.glob(f'{question_id}.*'))
        if not image_matches:
            raise FileNotFoundError(f'No image found for held-out question {question_id}')

        model.set_adapter('simpo_tuned')
        response = generate_response(model, image_matches[0], record['query'])

        row = {
            'question_id': question_id,
            'question': record['query'],
            'ground_truth': record['answer'],
            'responses': {system: response},
            'predicted_answers': {system: extract_final_answer(response)},
        }
        if idx == 0:
            if not response.strip():
                raise RuntimeError(
                    f'{system} produced an empty generation on holdout id={question_id}. '
                    'Aborting: likely collapsed LoRA or adapter load failure.'
                )
            print(f'Smoke OK for id={question_id}; sample response: {response[:200]!r}')
        handle.write(json.dumps(row, ensure_ascii=False) + '\n')
        handle.flush()
        done_ids.add(question_id)
        if (idx + 1) % 5 == 0 or idx == 0:
            print(f'[{idx + 1}/{len(eval_ids)}] completed question_id={question_id}')

print(f'Saved generations to {OUTPUT_PATH}')


In [ ]:
# Cell 5: Score with the exact same exact-match convention as evaluate_holdout.ipynb
rows = []
with OUTPUT_PATH.open(encoding='utf-8') as handle:
    for line in handle:
        line = line.strip()
        if line:
            rows.append(json.loads(line))

if len(rows) != len(eval_ids):
    raise AssertionError(f'Expected {len(eval_ids)} rows, found {len(rows)}')

system = 'simpo_tuned'
correct = 0
extracted = 0
for row in rows:
    pred = row['predicted_answers'].get(system, '')
    if pred:
        extracted += 1
    if normalize_answer(pred) == normalize_answer(row['ground_truth']):
        correct += 1

summary = {
    'n': len(rows),
    'exact_match': {system: {'correct': correct, 'accuracy': correct / len(rows)}},
    'extracted_answer_rate': {system: extracted / len(rows)},
}
SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(json.dumps(summary, indent=2))
print(f'Saved summary to {SUMMARY_PATH}')
